# Set up
## Environment Setup and Imports
This cell configures the Python path and imports the libraries used for hydrodynamics, data handling, and plotting.

In [ ]:
import sys
import os
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import pdstripy

import capytaine as cpy
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from wavespectra.construct.frequency import jonswap

import logging
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)

## Define Physical and Frequency Parameters
This cell sets fluid density, the angular-frequency sweep, and forward speed for the hydrodynamic analysis.

In [ ]:
rho = 1000
w = np.logspace(-1,np.log10(26),12)
forward_speed = 0

## Load Geometry and Extract Lid
This cell loads the spheroid mesh and separates hull and lid surfaces required by the solver.

In [ ]:
mesh = cpy.load_mesh("spheroid_6_1.stl")
hull_mesh, lid_mesh = mesh.extract_lid()

## Build Floating Body Model
This cell constructs the floating body, assigns rigid-body degrees of freedom, and computes inertia and hydrostatics.

In [ ]:
fb = cpy.FloatingBody(mesh=hull_mesh, 
                      name="6:1 spheroid",
                      center_of_mass=[0,0,-0.75],
                      lid_mesh=lid_mesh)
fb.add_all_rigid_body_dofs()
fb.inertia_matrix = fb.compute_rigid_body_inertia()
hs = fb.compute_hydrostatics()
# fb.show()

## Create Radiation and Diffraction Problems
This cell defines the set of boundary-element radiation and diffraction problems across all frequencies and relevant DOFs.

In [ ]:
problems = [
    cpy.RadiationProblem(body=fb, radiating_dof=dof, omega=omega, rho=rho, forward_speed=forward_speed)
    for dof in fb.dofs
    for omega in w
]
problems += [
    cpy.DiffractionProblem(body=fb, omega=omega, rho=rho, forward_speed=forward_speed, wave_direction=0*np.pi/180)
      for omega in w]

# Solve Capytaine
This cell runs the BEM solver for all defined problems and gathers the results into an xarray dataset.

In [ ]:
solver = cpy.BEMSolver()
results = solver.solve_all(problems)
ds = cpy.assemble_dataset(results)
ds

# Parse PDStrip Output Data
This cell reads PDStrip result files, reports warnings/errors, and loads the parsed dataset for comparison.

In [ ]:
parsed = pdstripy.parse_pdstrip_folder(os.path.join("pdstrip_run"))
pdstrip_ds = parsed["dataset"]
pdstrip_ds

## Visualize PD Section Geometry
This cell plots section points in the transverse plane to inspect the symmetry and layout of PDStrip sections.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for section in pdstrip_ds['section']:
    dsi = pdstrip_ds.sel(section=section)
    if dsi['x'] < 0:
        ss = -1
    else:
        ss = 1
    ax.plot(ss*dsi['y'], dsi['z'], marker='.', 
            label=f'Sec. {section.item()}: x={dsi["x"].item():.1f} m')
    
ax.axvline(0, color='k', ls='-', lw=0.5)

ax.legend(loc='lower left',ncols=2, bbox_to_anchor=(0,1), fontsize='small')


ax.spines[['top', 'right']].set_visible(False)
ax.set_xlabel("y [m]")
ax.set_ylabel("z [m]")

# Compare Heave Added Mass
This cell plots heave added-mass results from PDStrip and Capytaine on the same axes for direct comparison.

In [ ]:
fig, ax = plt.subplots()

pdstrip_ds['added_mass'].integrate('section').sel(influenced_dof='Heave', radiating_dof='Heave').plot(x='omega', ax=ax, label='PDstrip', marker='.')
ds['added_mass'][:,2,2].plot(ax=ax, label='capytaine', marker='.')
ax.set_xlabel('omega [rad/s]')

ax.legend()
ax.set_title("Heave")
ax.autoscale(enable=True, axis='x', tight=True)
ax.spines[['top', 'right']].set_visible(False)